> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports (active)

All chunking functions are imported from `src/mrta/`:

```python
from mrta.core.schemas import Chunk
from mrta.ingestion.chunker import chunk_pdf, fixed_chunks, recursive_chunks, token_chunks, semantic_chunks
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 02.

# Phase 2 — Chunking Strategies

**Goal:** Turn page-level text into retrievable chunks that (a) fit in an LLM context window, (b) carry enough surrounding context to be self-contained, and (c) preserve `{doc_id, page, section, chunk_id}` for citations.

We compare three strategies:

1. **Fixed-size character chunking** — naive baseline.
2. **Recursive character splitting** — LangChain's default. Splits on paragraph/sentence/word boundaries.
3. **Token-aware splitting** — uses `tiktoken` so chunks fit a known token budget.

We will see why **chunk size 500–800 tokens with ~100-token overlap** is the canonical sweet spot for research papers.


## 2.1 — Load the parsed PDF from Phase 1


In [1]:
import os, sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from mrta.core.schemas import PageRecord, PdfDocument
from mrta.ingestion import load_pdf

pdf = load_pdf("data/sample/attention_is_all_you_need.pdf")
print(f"Loaded {pdf.n_pages} pages from {pdf.source}")

Loaded 15 pages from attention_is_all_you_need.pdf


## 2.2 — Define the chunk schema

A chunk is the smallest indexable unit in our system. Every chunk knows where it came from.


In [2]:
from mrta.core.schemas import Chunk

## 2.3 — Strategy A: Fixed-size character chunking

The naive baseline. Just cut every N characters with M overlap. Cheap, but it splits sentences mid-word and ignores paragraph structure.


In [3]:
from mrta.ingestion.chunker import fixed_chunks

fixed = fixed_chunks(pdf)
print(f"Fixed-size: {len(fixed)} chunks")
print("Example chunk preview:\n")
print(fixed[3].text[:300], "\n...")

Fixed-size: 38 chunks
Example chunk preview:

1
Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numero 
...


## 2.4 — Strategy B: Recursive character splitting

LangChain's `RecursiveCharacterTextSplitter` tries a sequence of separators (`\n\n`, `\n`, `. `, ` `, `""`) and falls back to finer ones only if the chunk is still too large. Result: clean paragraph/sentence boundaries.


In [4]:
from mrta.ingestion.chunker import recursive_chunks

recursive = recursive_chunks(pdf)
print(f"Recursive: {len(recursive)} chunks")
print("\nExample:\n", recursive[5].text[:300])

Recursive: 61 chunks

Example:
 sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through factorization tricks [21] and condition


## 2.5 — Strategy C: Token-aware splitting

`tiktoken` lets us count tokens the same way the LLM will. This matters because context windows are measured in tokens, not characters, and the character-to-token ratio varies between models (English prose is ~4 chars/token).


In [5]:
from mrta.ingestion.chunker import token_chunks

tokenized = token_chunks(pdf)
print(f"Token-aware: {len(tokenized)} chunks")
sizes = [c.n_tokens for c in tokenized]
print(f"Token sizes: min={min(sizes)}, median={sorted(sizes)[len(sizes)//2]}, max={max(sizes)}")

Token-aware: 25 chunks
Token sizes: min=66, median=394, max=700


## 2.6 — Side-by-side comparison


In [6]:
import pandas as pd

def summarize(name: str, chunks: list[Chunk]) -> dict:
    char_lens = [len(c.text) for c in chunks]
    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "median_chars": int(pd.Series(char_lens).median()),
        "p95_chars": int(pd.Series(char_lens).quantile(0.95)),
        "first_chunk_starts_clean": chunks[0].text[:1].isupper(),
    }

pd.DataFrame([
    summarize("fixed-1500/200",      fixed),
    summarize("recursive-800/100",   recursive),
    summarize("token-700/100",       tokenized),
])


,strategy,n_chunks,median_chars,p95_chars,first_chunk_starts_clean
0,fixed-1500/200,38,1500,1500,True
1,recursive-800/100,61,745,793,True
2,token-700/100,25,1411,3063,True


## 2.7 — Why these defaults?

| Parameter      | Default       | Reasoning                                                                                          |
|----------------|---------------|----------------------------------------------------------------------------------------------------|
| `chunk_size`   | 500–800 tokens| Large enough to contain a complete idea (a paragraph or two), small enough to leave room for ~3–5 retrieved chunks + the question + the answer in a 4k–8k context window. |
| `overlap`      | ~15% of size  | Lets information that straddles a chunk boundary appear in both, so retrieval is more robust.       |
| Separator order| `\n\n` → `\n` → `. ` → ` ` | Paragraph > line > sentence > word. Aim to break at semantic boundaries first.    |

If you swap to a model with a long context (e.g. 128k), you can go up to 1500–2000 token chunks and retrieve fewer of them. But beware: longer chunks dilute embedding-based retrieval ("lost in the middle" effect).


## 2.8 — Semantic chunking (advanced)

The strategies above are *structural*. A more advanced approach is **semantic chunking**: greedily merge adjacent sentences as long as their embeddings stay similar, then start a new chunk when similarity drops. This produces topic-coherent chunks.

We sketch it here using sentence embeddings; we will not make it the default because it is slower and benefits saturate around 700-token chunks for short papers.


In [7]:
from mrta.ingestion.chunker import semantic_chunks

# semantic_chunks is slower — run only the first 5 pages for the demo.
import copy
demo_pdf = pdf.model_copy(deep=True)
demo_pdf.pages = pdf.pages[:5]
semantic = semantic_chunks(demo_pdf)
print(f"Semantic: {len(semantic)} chunks over first 5 pages")
print("\nExample:\n", semantic[2].text[:300])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic: 84 chunks over first 5 pages

Example:
 Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decode


## 2.9 — Save the canonical chunks for downstream notebooks

We will use the **token-aware** chunks (`token_chunks`, size=700, overlap=100) as the default for Phase 3 onwards. Saving them to `data/processed/` so we do not have to recompute.


In [8]:
import json
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

chunks_path = processed_dir / "chunks.jsonl"
with chunks_path.open("w", encoding="utf-8") as f:
    for c in tokenized:
        f.write(c.model_dump_json() + "\n")

print(f"Saved {len(tokenized)} chunks to {chunks_path}")
print(f"File size: {chunks_path.stat().st_size/1024:.1f} KB")


Saved 25 chunks to data/processed/chunks.jsonl
File size: 48.3 KB


## What you learned

- Three chunking strategies and their tradeoffs.
- Why token-aware splitting is the production default.
- The `Chunk` schema with `chunk_id`, `page`, `doc_id` for citations.
- A semantic chunking sketch for advanced use.
- How to persist chunks to JSONL.

## Exercises

1. Implement section-aware chunking: parse Markdown-like headings (`Abstract`, `Introduction`, `Methods`) and never let a chunk straddle a section boundary.
2. Plot a histogram of chunk character lengths for each strategy.
3. Tune `chunk_size` to 400, 800, 1200 — which is best? You will revisit this in Phase 9 with real eval metrics.

## Next: [Phase 3 — Embeddings & FAISS](./2026-05-25-phase03-embeddings-and-faiss.ipynb)


# Answer 1. Implement section-aware chunking: parse Markdown-like headings (`Abstract`, `Introduction`, `Methods`) and never let a chunk straddle a section boundary.
# 
# 
# The problem with existing chunkers

All four existing strategies in chunker.py operate page-by-page and split text purely by size. A recursive_chunks call on a page containing the end of Introduction and the start of Methods will produce a chunk like:


"...building on prior work in neural machine translation.

Methods

We train on the WMT 2014 English-German dataset..."
That chunk spans two sections — a query about "methods" retrieves it, but it's polluted with unrelated introduction content. Section-aware chunking prevents this.

The approach: split first, then chunk within


Full document text
       │
       ▼
  _parse_sections()          ← find heading positions, split text
       │
       ▼
[("Abstract", "We propose..."),
 ("Introduction", "Neural networks..."),
 ("Methods", "We train on..."),
 ...]
       │
       ▼
  chunk each section independently  ← no chunk crosses a boundary
       │
       ▼
  Chunk(text=..., section="Methods", ...)
Implementation to add to chunker.py


# Section heading patterns common in NLP/CV papers
_HEADING_RE = re.compile(
    r"(?m)^[ \t]*(\d+\.?\d*\.?\s+)?("
    r"Abstract|Introduction|Background|Related Work|"
    r"Methodology?|Experiments?|Results?|"
    r"Discussion|Conclusion|References?"
    r")\s*$",
    re.IGNORECASE,
)


def _parse_sections(text: str) -> list[tuple[str, str]]:
    """Return [(section_title, section_text), ...] split at heading lines.

    Text before the first heading is labelled 'preamble'.
    """
    matches = list(_HEADING_RE.finditer(text))
    if not matches:
        return [("body", text)]

    sections: list[tuple[str, str]] = []
    # Text before first heading
    if matches[0].start() > 0:
        sections.append(("preamble", text[: matches[0].start()]))

    for i, m in enumerate(matches):
        title = m.group(0).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            sections.append((title, body))

    return sections


def section_chunks(
    pdf: PdfDocument,
    size: int = 800,
    overlap: int = 100,
) -> list[Chunk]:
    """Chunk the document respecting section boundaries.

    Uses RecursiveCharacterTextSplitter within each section so chunks
    never straddle a heading. The section title is prepended to each
    chunk so the retriever knows where it came from.
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    # Combine all pages into one document string, tracking page boundaries
    full_text = "\n\n".join(p.text for p in pdf.pages)
    doc_id = pdf.doc_id
    source = pdf.source

    sections = _parse_sections(full_text)
    out: list[Chunk] = []
    chunk_idx = 0

    for section_title, section_text in sections:
        for piece in splitter.split_text(section_text):
            if not piece.strip():
                continue
            out.append(
                Chunk(
                    chunk_id=f"{doc_id}_s{chunk_idx}",
                    doc_id=doc_id,
                    source=source,
                    page=1,          # section may span pages — best effort
                    text=f"[{section_title}]\n{piece}",
                )
            )
            chunk_idx += 1

    return out
Then register it in _STRATEGIES:


_STRATEGIES: dict[str, object] = {
    "fixed": fixed_chunks,
    "recursive": recursive_chunks,
    "token": token_chunks,
    "semantic": semantic_chunks,
    "section": section_chunks,      # ← add this
}
What a resulting chunk looks like


[Methods]
We train on the WMT 2014 English-German dataset consisting of
about 4.5 million sentence pairs. Sentences were encoded using
byte-pair encoding with a shared vocabulary of 37,000 tokens.
The [Methods] prefix travels with the chunk into the vector store — the embedder sees it, and the retriever can filter on it too.

Known limitation: page number is approximate

Because _parse_sections works on the concatenated full-text string, the per-chunk page field loses precision (sections can span multiple pages). To fix this properly you'd track character offsets back to PageRecord boundaries — worth a follow-up if precise page attribution matters for citations.

In [ ]:
### Answer 2. Plot a histogram of chunk character lengths for each strategy.

import matplotlib.pyplot as plt
from mrta.ingestion.pdf_loader import load_pdf
from mrta.ingestion.chunker import fixed_chunks, recursive_chunks, token_chunks

pdf = load_pdf("tests/fixtures/sample.pdf")

strategies = {
    "fixed":     fixed_chunks(pdf, size=1500, overlap=200),
    "recursive": recursive_chunks(pdf, size=800, overlap=100),
    "token":     token_chunks(pdf, size=700, overlap=100),
}

fig, axes = plt.subplots(1, len(strategies), figsize=(14, 4), sharey=True)

for ax, (name, chunks) in zip(axes, strategies.items()):
    lengths = [len(c.text) for c in chunks]
    ax.hist(lengths, bins=20, edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("chunk length (chars)")
    ax.axvline(sum(lengths) / len(lengths), color="red", linestyle="--", label=f"mean={sum(lengths)//len(lengths)}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("count")
plt.suptitle("Chunk length distribution by strategy")
plt.tight_layout()
plt.show()
What to look for in the output:

Strategy	Expected shape
fixed	Narrow spike — nearly all chunks the same length (by design)
recursive	Slightly wider spread — respects paragraph breaks so chunks vary a little
token	Similar to fixed but measured in tokens, not chars — you'll notice shorter char-lengths for token-dense text
The red dashed line marks the mean. If recursive shows a long left tail (many short chunks), that means the PDF has a lot of short paragraphs — headers, captions, bullet points — that don't fill a full chunk window.

3. Tune `chunk_size` to 400, 800, 1200 — which is best? You will revisit this in Phase 9 with real eval metrics.

There's no universally correct answer — it depends on your query type. Here's how to think about it and measure it.

The tradeoff

chunk_size	Precision	Recall	Risk
400	High — tight, focused	Low — answer may be split across chunks	Over-fragmentation: one idea split in two
800	Balanced	Balanced	Usually the safe default
1200	Low — lots of surrounding context	High — answer is likely inside	Noise: retrieves too much irrelevant text
Smaller chunks = precise retrieval but risk splitting a concept across two chunks.
Larger chunks = capture full ideas but dilute the embedding signal with surrounding noise.

Code to compare all three empirically


from mrta.ingestion.pdf_loader import load_pdf
from mrta.ingestion.chunker import recursive_chunks

pdf = load_pdf("tests/fixtures/sample.pdf")

for size in [400, 800, 1200]:
    chunks = recursive_chunks(pdf, size=size, overlap=size // 8)
    lengths = [len(c.text) for c in chunks]
    print(f"size={size:4d}  →  {len(chunks):3d} chunks  "
          f"mean={sum(lengths)//len(lengths):4d} chars  "
          f"min={min(lengths):3d}  max={max(lengths):4d}")
How to decide which is best for your use case

Run a small retrieval experiment against your golden QA dataset. For each chunk size, embed the chunks, then for each question retrieve top-5 and check if the answer is inside:


from mrta.retrieval.vector_store import VectorStore
from mrta.retrieval.embedder import Embedder
from mrta.evaluation.metrics import recall_at_k

embedder = Embedder()
results = {}

for size in [400, 800, 1200]:
    chunks = recursive_chunks(pdf, size=size, overlap=size // 8)
    vs = VectorStore(embedder)
    vs.add(chunks)

    # ask a few questions manually and check what comes back
    hits = vs.search("What is self-attention?", k=5)
    results[size] = [c.text[:120] for c in hits]
    print(f"\n--- size={size} ---")
    for h in results[size]:
        print(" •", h)
Rule of thumb for academic RAG:

Short factual questions ("What dataset was used?") → 400–600
Conceptual questions ("How does the attention mechanism work?") → 800–1000
Summary/synthesis questions ("What are the main contributions?") → 1000–1400
Most production systems use 800 with ~10% overlap as the starting point, then tune based on retrieval eval metrics.